# 📗 Kit 2 — Modélisation & Pipeline ETL
## Formation Data Engineer

---

### Comment utiliser ce notebook ?

- **`# ✅ CODE FOURNI`** → lisez attentivement et exécutez
- **`# 🔧 TODO`** → vous devez compléter ce bloc

### Prérequis
- Avoir `dataset_final_kit1.csv` dans le même dossier

### Livrables attendus
- `memo_formes_normales.md`
- `data/clean/communes.csv`, `secteurs.csv`, `entreprises.csv`
- `pipeline.py` + `pipeline.log`
- `entreprises_isere.db`


In [ ]:
# ✅ CODE FOURNI — Imports
import pandas as pd
import sqlite3
import logging
import json
import os
import shutil

os.makedirs('data/clean', exist_ok=True)
os.makedirs('data/raw',   exist_ok=True)

if os.path.exists('dataset_final_kit1.csv') and not os.path.exists('data/raw/dataset_final_kit1.csv'):
    shutil.copy('dataset_final_kit1.csv', 'data/raw/dataset_final_kit1.csv')

SRC = 'data/raw/dataset_final_kit1.csv'

if not os.path.exists(SRC):
    print(f'❌ {SRC} introuvable — terminez le Kit 1 d\'abord')
else:
    df = pd.read_csv(SRC)
    print(f'✓ Dataset chargé : {len(df)} lignes, {len(df.columns)} colonnes')
    print(f'Colonnes : {df.columns.tolist()}')


---
## Section 4 — Partie 4 : Analyse & Modélisation UML

### Objectif
Analyser le dataset avec Python pour identifier les redondances et déterminer quelles entités doivent être séparées dans votre diagramme UML.

### Qu'est-ce qu'une redondance ?
La latitude et longitude de Grenoble sont copiées dans TOUTES les lignes des entreprises de Grenoble. Dans un bon modèle, ces informations ne sont stockées qu'**une seule fois** dans une table Commune.


In [ ]:
# 🔧 TODO — Exercice 4.1 : Analyser la cardinalité
#
# 1. Affichez le nombre de valeurs uniques par colonne
# 2. Quelles colonnes ont beaucoup de répétitions ?

pd.options.display.max_columns = None
print('=== 5 premières lignes ===')
print(df.head(3).to_string())

print('\n=== Cardinalité (valeurs uniques par colonne) ===')
for col in df.columns:
    print(f'  {col:25s} : ??? valeurs uniques')  # TODO : df[col].nunique()


In [ ]:
# ✅ CODE FOURNI — Analyse des redondances
# Objectif : identifier quelles colonnes se répètent inutilement
# et en déduire quelles entités gagneraient à être séparées dans le modèle.

print('=== ANALYSE DES REDONDANCES ===')

# Les coords GPS sont-elles constantes pour une commune ?
# Si oui → elles appartiennent à Commune, pas à Entreprise
check = df.groupby('nom_commune')[['latitude','longitude']].nunique()
print(f'Communes avec coords incohérentes : {len(check[check.max(axis=1) > 1])}')
print('  → lat/lon se répètent par commune mais restent identiques')
print('  → candidat : table Commune séparée')

# Un SIREN identifie-t-il une seule entreprise ?
siren_multi = df.groupby('siren')['nom_commune'].nunique()
print(f'\nSIREN dans plusieurs communes : {(siren_multi > 1).sum()}')
print('  → siren identifie une entreprise unique : candidat PK de Entreprise')

# Le code_naf est-il partagé entre plusieurs entreprises ?
print(f'\nNombre de code_naf distincts : {df["code_naf"].nunique()}')
print(f'Entreprises par code_naf en moyenne : {len(df) / max(df["code_naf"].nunique(),1):.1f}')
print('  → code_naf partagé entre entreprises : candidat à une table Secteur séparée')

print('\n=== OBSERVATIONS ===')
print('3 groupes de données redondantes identifiés :')
print('  1. Informations géographiques de la commune (lat, lon, code_postal...)')
print('  2. Informations sur le secteur d\'activité (code_naf, libelle_naf...)')
print('  3. Informations sur la tranche d\'effectif (code_tranche, libelle_tranche...)')
print()
print('→ Ces observations vont guider votre diagramme UML.')
print('  Les formes normales (Section 5) formaliseront ces intuitions.')
print()
print('📌 Créez maintenant votre diagramme UML v1 sur draw.io — sauvegarder en diagramme_uml.png')


In [ ]:
# 🔧 TODO — Q2, Q3 & Q4 : Préparer le diagramme UML
#
# Q2. Si le code postal de Grenoble change, combien de lignes
#     faut-il modifier dans le dataset plat ?
#     Et dans un modèle où Commune est une table séparée ?
#     Répondez en commentaire.
#
# Q3. Identifiez au moins 3 entités distinctes dans le dataset.
#     Listez leurs attributs en commentaire.
#
# Q4. Créez le diagramme UML v1 sur draw.io ou Excalidraw.
#     Sauvegardez en diagramme_uml.png.

# TODO Q2 :
# Dataset plat    : ???
# Table Commune   : ???

# TODO Q3 — entités et attributs :
# Entité 1 : ???
# Entité 2 : ???
# Entité 3 : ???

# TODO Q4 : diagramme à créer sur draw.io — sauvegarder en diagramme_uml.png
print('Q4 : Créez votre diagramme UML v1 sur draw.io ou Excalidraw')


---
## Section 5 — Partie 5 : Formes normales & Scripts de transformation

### Rappel des formes normales

**1NF** : chaque cellule = une valeur atomique (pas de listes). Chaque ligne est unique.

**2NF** : en 1NF + tous les attributs dépendent de la TOTALITÉ de la clé primaire. Ne s'applique qu'aux clés composites. Si PK = siren (une colonne), 1NF implique automatiquement 2NF.

**3NF** : en 2NF + pas de dépendance transitive. `siren → code_naf → libelle_naf` est une violation. Solution : table Secteur séparée.


In [ ]:
# 🔧 TODO — Exercice 5.1 : Rédiger le mémo des formes normales
#
# Complétez les ??? et sauvegardez dans memo_formes_normales.md

memo_lines = [
    '# Mémo : Formes Normales\n',
    '## Appliquées au dataset entreprises Isère\n',
    '\n',
    '## 1NF\n',
    'Définition : ???\n',
    'Exemple de violation dans notre dataset : ???\n',
    'Notre dataset est-il en 1NF ? ???\n',
    '\n',
    '## 2NF\n',
    'Définition : ???\n',
    'À quoi s\'applique-t-elle ? ???\n',
    'Notre dataset est-il en 2NF ? ???\n',
    '\n',
    '## 3NF\n',
    'Définition : ???\n',
    'Violation dans notre dataset : ???\n',
    'Solution appliquée : ???\n',
    '\n',
    '## Conclusion\n',
    'À quelle forme normale correspond le dataset plat ? ???\n',
    'À quelle forme normale correspond notre modèle cible (3 tables) ? ???\n',
]

with open('memo_formes_normales.md', 'w', encoding='utf-8') as f:
    f.writelines(memo_lines)
print('✓ memo_formes_normales.md créé (à compléter !)')


### Exercice 5.2 — Scripts de transformation

Voici `extract_communes()` fourni comme modèle. Lisez attentivement avant d'écrire les deux autres.


In [ ]:
# ✅ CODE FOURNI — Q8 : extract_communes() — modèle de référence
# Lisez chaque commentaire attentivement avant d'écrire les deux suivants.

def extract_communes(src, output='data/clean/communes.csv'):
    df = pd.read_csv(src)
    # drop_duplicates : une ligne par commune (le dataset plat a N lignes par commune)
    c = df[['nom_commune','code_postal','latitude','longitude']]\
         .drop_duplicates('nom_commune').copy()
    c.columns = ['nom','code_postal','latitude','longitude']
    # Code INSEE simulé : 38 (Isère) + numéro séquentiel
    c.insert(0, 'code_insee', [f'38{i:03d}' for i in range(1, len(c)+1)])
    c.to_csv(output, index=False)  # index=False : pas de colonne numérique inutile
    print(f'extract_communes : {len(c)} communes → {output}')
    return c

df_communes = extract_communes(SRC)
print(df_communes.head(3).to_string())


In [ ]:
# 🔧 TODO — Q9 : extract_secteurs() et extract_tranches()
#
# Sur le modèle de extract_communes(), écrivez extract_secteurs() qui :
# - extrait les colonnes code_naf ET libelle_naf (les deux sont maintenant dans le dataset)
# - dédoublonne sur code_naf
# - supprime les lignes sans code_naf
# - sauvegarde dans data/clean/secteurs.csv
#
# Puis écrivez extract_tranches() sur le même modèle :
# - extrait code_tranche ET libelle_tranche
# - dédoublonne sur code_tranche
# - sauvegarde dans data/clean/tranches_effectifs.csv

def extract_secteurs(src, output='data/clean/secteurs.csv'):
    df = pd.read_csv(src)
    s = ???  # TODO
    s.to_csv(???, index=False)
    print(f'extract_secteurs : {len(s)} → {output}')
    return s

def extract_tranches(src, output='data/clean/tranches_effectifs.csv'):
    df = pd.read_csv(src)
    t = ???  # TODO
    t.to_csv(???, index=False)
    print(f'extract_tranches : {len(t)} → {output}')
    return t

df_secteurs  = extract_secteurs(SRC)
df_tranches  = extract_tranches(SRC)
print(df_secteurs.head())
print(df_tranches.head())


In [ ]:
# 🔧 TODO — Q10 & Q11 : extract_entreprises()
#
# Q10. Écrivez extract_entreprises() :
#      - normalise nom_commune des deux côtés avant le merge
#      - merge LEFT pour obtenir le code_insee
#      - sélectionne : siren, nom, code_naf, code_insee, code_tranche, adresse
#        (on garde code_tranche comme FK vers tranches_effectifs)
#      - dédoublonne sur siren, supprime les lignes sans siren
#
# Q11. Testez le script et vérifiez les résultats avant de continuer.

def extract_entreprises(src, communes_path, output='data/clean/entreprises.csv'):
    df  = pd.read_csv(src)
    com = pd.read_csv(communes_path)

    com['nom_clean'] = ???
    df['nom_clean']  = ???

    df_merge = df.merge(???, on='nom_clean', how='left')

    taux = df_merge['code_insee'].isnull().mean() * 100
    print(f'  Entreprises sans code_insee : {taux:.1f}%')

    # code_tranche (FK vers tranches_effectifs) remplace tranche_effectif
    e = df_merge[['siren', 'nom', 'code_naf', 'code_insee', 'code_tranche', 'adresse']].copy()
    e = e.drop_duplicates(subset=[???])
    e = e.dropna(subset=[???])

    e.to_csv(output, index=False)
    print(f'extract_entreprises : {len(e)} → {output}')
    return e

df_entreprises = extract_entreprises(SRC, 'data/clean/communes.csv')
print(df_entreprises.head(3).to_string())


---
## Section 6 — Partie 6 : Pipeline ETL & SQLite

### Le pipeline ETL
1. **Extract** : lire les données brutes
2. **Transform** : appeler nos fonctions extract_*
3. **Load** : charger dans SQLite

### SQLite : base de données dans un fichier
SQLite stocke tout dans un fichier `.db`. Python intègre sqlite3 en standard.

**Les 3 lignes obligatoires :**
```python
conn = sqlite3.connect('ma_base.db')  # Crée ou ouvre
conn.commit()                          # OBLIGATOIRE : persiste les changements
conn.close()                           # Ferme la connexion
```

⚠️ Sans `conn.commit()`, les insertions sont en mémoire et perdues !


In [ ]:
# ✅ CODE FOURNI — load_sqlite() avec table de rejets et tranches_effectifs

DDL = '''
DROP TABLE IF EXISTS rejected_rows;
DROP TABLE IF EXISTS entreprises;
DROP TABLE IF EXISTS tranches_effectifs;
DROP TABLE IF EXISTS communes;
DROP TABLE IF EXISTS secteurs;

CREATE TABLE communes(
    code_insee  TEXT PRIMARY KEY,
    nom         TEXT NOT NULL,
    code_postal TEXT,
    latitude    REAL,
    longitude   REAL
);

CREATE TABLE secteurs(
    code_naf    TEXT PRIMARY KEY,
    libelle_naf TEXT
);

CREATE TABLE tranches_effectifs(
    code_tranche    TEXT PRIMARY KEY,
    libelle_tranche TEXT
);

CREATE TABLE entreprises(
    siren            TEXT PRIMARY KEY,
    nom              TEXT,
    code_naf         TEXT REFERENCES secteurs(code_naf),
    code_insee       TEXT REFERENCES communes(code_insee),
    code_tranche     TEXT REFERENCES tranches_effectifs(code_tranche),
    adresse          TEXT
);

CREATE TABLE rejected_rows(
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    table_name TEXT,
    row_data   TEXT,
    error_msg  TEXT,
    ts         TEXT DEFAULT (datetime('now'))
);
'''

def safe_insert(cursor, table, df):
    ok = err = 0
    for _, row in df.iterrows():
        d = {k: (None if pd.isna(v) else v) for k, v in row.items()}
        try:
            cols = ','.join(d.keys())
            ph   = ','.join(['?'] * len(d))
            cursor.execute(f'INSERT OR IGNORE INTO {table}({cols}) VALUES({ph})', list(d.values()))
            ok += 1
        except Exception as e:
            cursor.execute(
                'INSERT INTO rejected_rows(table_name,row_data,error_msg) VALUES(?,?,?)',
                [table, str(d), str(e)]
            )
            err += 1
    print(f'  {table:25s} : {ok:5d} insérées, {err:3d} rejetées')

def load_to_sqlite(db_path, df_communes, df_secteurs, df_tranches, df_entreprises):
    conn = sqlite3.connect(db_path)
    conn.executescript(DDL)
    cur  = conn.cursor()
    # Ordre important : charger les tables référencées avant entreprises
    safe_insert(cur, 'communes',          df_communes)
    safe_insert(cur, 'secteurs',          df_secteurs)
    safe_insert(cur, 'tranches_effectifs', df_tranches)
    safe_insert(cur, 'entreprises',       df_entreprises)
    conn.commit()
    conn.close()
    print(f'✓ Base SQLite créée : {db_path}')

print('✓ Fonctions définies')


In [ ]:
# 🔧 TODO — Q12, Q13 & Q14 : Logique du pipeline
#
# Q12. Quelle approche avons-nous utilisée (ETL ou ELT) ? Justifiez en 2 phrases.
#
# Q13. Écrivez run_pipeline() qui orchestre les 3 étapes avec logging.
#      Attention : load_to_sqlite attend maintenant 5 arguments
#      (db, communes, secteurs, tranches, entreprises).
#
# Q14. load_to_sqlite est fourni. Expliquez en commentaire :
#      - pourquoi OR IGNORE dans les INSERT ?
#      - à quoi sert la table rejected_rows ?
#      - pourquoi charger tranches_effectifs avant entreprises ?

# TODO Q12 :
# Approche : ???
# Justification : ???

# TODO Q14 :
# OR IGNORE : ???
# rejected_rows : ???
# Ordre d'insertion : ???

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('pipeline.log', encoding='utf-8')
    ]
)
log = logging.getLogger('pipeline')
DB  = 'entreprises_isere.db'

def run_pipeline():
    log.info('=== DEBUT PIPELINE ETL ===')

    log.info('[EXTRACT] Chargement...')
    df_raw = ???
    log.info(f'[EXTRACT] ??? lignes')

    log.info('[TRANSFORM] Communes...')
    df_c = ???
    log.info('[TRANSFORM] Secteurs...')
    df_s = ???
    log.info('[TRANSFORM] Tranches effectifs...')
    df_t = ???
    log.info('[TRANSFORM] Entreprises...')
    df_e = ???

    log.info(f'[TRANSFORM] {len(df_c)} communes, {len(df_s)} secteurs, '
             f'{len(df_t)} tranches, {len(df_e)} entreprises')

    log.info(f'[LOAD] Chargement dans {DB}...')
    ???  # load_to_sqlite(DB, df_c, df_s, df_t, df_e)

    log.info('=== PIPELINE TERMINÉ ===')

run_pipeline()


In [ ]:
# 🔧 TODO — Exercice 6.3 : Requêtes de validation SQLite
#
# 1. Comptez les lignes de chaque table
# 2. Vérifiez l'intégrité référentielle
# 3. Affichez le top 5 des communes par nombre d'entreprises

conn = sqlite3.connect(DB)
cur  = conn.cursor()

print('=== RAPPORT DE VALIDATION ===')

# TODO 1 : comptages
for table in ['communes', 'secteurs', 'entreprises', 'rejected_rows']:
    cur.execute(f'SELECT COUNT(*) FROM {table}')
    print(f'  {table:20s} : {cur.fetchone()[0]}')

# TODO 2 : intégrité (entreprises avec code_insee absent de communes)
cur.execute('''
    SELECT COUNT(*) FROM entreprises
    WHERE code_insee IS NOT NULL
    AND code_insee NOT IN (SELECT code_insee FROM communes)
''')
print(f'\nEntreprises avec code_insee invalide : {cur.fetchone()[0]}')

# TODO 3 : top 5 communes
# Hint : SELECT c.nom, COUNT(e.siren) AS nb FROM communes c LEFT JOIN entreprises e ...
cur.execute('''
    ???
''')
print('\nTop 5 communes :')
for nom, nb in cur.fetchall():
    print(f'  {nom:30s} : {nb}')

conn.close()


In [ ]:
# ✅ CODE FOURNI — Consulter les rejected_rows

conn = sqlite3.connect(DB)
df_rej = pd.read_sql('SELECT * FROM rejected_rows', conn)
conn.close()

print(f'Total lignes rejetées : {len(df_rej)}')
if len(df_rej) > 0:
    print(df_rej[['table_name','error_msg']].head(5).to_string())
else:
    print('✓ Aucune ligne rejetée')


---
## ✅ Bilan du Kit 2

| Fichier | Description |
|---|---|
| `diagramme_uml.png` + `diagramme_uml_3NF.png` | Modèles UML |
| `memo_formes_normales.md` | Analyse 1NF/2NF/3NF |
| `data/clean/communes.csv` | Table communes |
| `data/clean/secteurs.csv` | Table secteurs (avec libellés) |
| `data/clean/tranches_effectifs.csv` | Table tranches effectifs |
| `data/clean/entreprises.csv` | Table entreprises |
| `pipeline.log` | Trace d'exécution |
| `entreprises_isere.db` | Base SQLite — 5 tables |

**Félicitations — pipeline data engineer complet !**
